In [1]:
import pandas as pd
from pyspark.sql import functions as F

from pyspark.sql.types import DoubleType, StringType
import re
from functools import reduce
from operator import add

In [2]:
from pyspark.sql import SparkSession
from src.utils.logger import get_logger
import src.utils.config as config 


print(f"DEBUG: Access Key is {config.MINIO_ACCESS_KEY[:3]} + ***") 
print(f"DEBUG: ENDPOINT is {config.MINIO_ENDPOINT}")

# Get the container's hostname dynamically

logger = get_logger(__name__)

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a configured Spark session for MinIO.
    """
    logger.info(f"Creating Spark Session: {app_name}")
    
    # This pulls the necessary S3A connectors from Maven Central
    
    

    spark = (   
        SparkSession.builder
        .appName(app_name)
        .master("spark://spark-master:7077")
        .config("spark.driver.host", config.DRIVER_HOST)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", config.SPARK_DRIVER_PORT)
        .config("spark.driver.blockManager.port", config.SPARK_BLOCK_MANAGER_PORT)
        .config("spark.sql.shuffle.partitions", "50")
        .config("spark.executor.instances", "1") # adjust based on resources
        .config("spark.executor.cores", "2")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "false")
        #.config("spark.executor.memory", "2g") # adjust based on resources
        #.config("spark.driver.memory", "2g") # adjust based on resources

         # hadoop S3A Configuration
      
        .config("spark.hadoop.fs.s3a.endpoint", config.MINIO_ENDPOINT)
        .config("spark.hadoop.fs.s3a.access.key", config.MINIO_ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", config.MINIO_SECRET_KEY)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(config.MINIO_SECURE).lower())
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # prevent class resolution
        .config("spark.sql.caseSensitive", "false")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
        .config("spark.cores.max", "2")
        .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
        .config("spark.hadoop.fs.s3a.fast.upload", "true") #performance improvement
        .config("spark.sql.files.maxPartitionBytes", "16777216") #16MB partitions
        #.config("spark.network.timeout", "1200s")
        #.config("spark.rpc.askTimeout", "600s")
        #.config("spark.executor.heartbeatInterval", "120s")
        #.config("spark.hadoop.fs.s3a.connection.timeout", "600000")
        #.config("spark.hadoop.fs.s3a.paging.maximum", "1000")
        
        .getOrCreate()
    )
    # Suppress verbose logs
    spark.sparkContext.setLogLevel("WARN")
    logger.info("Spark session created successfully")
    return spark

[CONFIG] Stage: transform
[CONFIG] Loaded env: /opt/spark-app/env/.env.transform
[CONFIG] MinIO Endpoint: lakehouse-minio:9000
[CONFIG] Access Key (masked): tra***
DEBUG: Access Key is tra + ***
DEBUG: ENDPOINT is lakehouse-minio:9000


In [3]:
spark= create_spark_session("notebook_test")
Germany_23= spark.read.parquet("s3a://bronze/GERMANY/2023_BRONZE/")

2026-07-27 22:14:55 | INFO | lakehouse.__main__ | Creating Spark Session: notebook_test


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/27 22:15:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/27 22:15:48 WARN TransportClientFactory: DNS resolution succeed for spark-master/172.18.0.3:7077 took 2425 ms


2026-07-27 22:16:03 | INFO | lakehouse.__main__ | Spark session created successfully


26/07/27 22:16:25 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
Germany_23.rdd.getNumPartitions()

3

In [5]:
Germany_23

DataFrame[haushaltsjahr: string, name_des_begã¼nstigten_rechtstrã¤gers_verdands: string, wenn_teil_einer_gruppe:_name_des_mutterunternehmens: string, wenn_teil_einer_gruppe:_steuerliches_identifikationsmerkmal: string, plz: string, gemeinde: string, betroffener_staat: string, code_der_maãnahme_der_interventionskategorie_des_sektors_gemã¤ã_anhang_ix: string, spezifisches_ziel: string, anfangsdatum: string, enddatum: string, betrag_je_vorhaben_im_rahmen_des_egfl: string, egfl__gesamtbetrag_fã¼r_diesen_begã¼nstigten: string, betrag_je_vorhaben_im_rahmen_des_eler_(eu_mittel): string, eler_gesamtbetrag_fã¼r_diesen_begã¼nstigten_(eu_mittel): string, betrag_je_vorhaben_im_rahmen_der_nationalen_kofinanzierung: string, national_kofinanzierter_gesamtbetrag_fã¼r_diesen_begã¼nstigten: string, summe_des_eler_betrags_(eu_mittel)_und_des_kofinanzierten_betrags: string, eu_betrag_(egfl_und_eler)_und_kofinanzierter_betrag_insgesamt_fã¼r_diesen_begã¼nstigten*: string, source_country: string, source_ye

In [7]:
Germany_23.printSchema()

root
 |-- haushaltsjahr: string (nullable = true)
 |-- name_des_begã¼nstigten_rechtstrã¤gers_verdands: string (nullable = true)
 |-- wenn_teil_einer_gruppe:_name_des_mutterunternehmens: string (nullable = true)
 |-- wenn_teil_einer_gruppe:_steuerliches_identifikationsmerkmal: string (nullable = true)
 |-- plz: string (nullable = true)
 |-- gemeinde: string (nullable = true)
 |-- betroffener_staat: string (nullable = true)
 |-- code_der_maãnahme_der_interventionskategorie_des_sektors_gemã¤ã_anhang_ix: string (nullable = true)
 |-- spezifisches_ziel: string (nullable = true)
 |-- anfangsdatum: string (nullable = true)
 |-- enddatum: string (nullable = true)
 |-- betrag_je_vorhaben_im_rahmen_des_egfl: string (nullable = true)
 |-- egfl__gesamtbetrag_fã¼r_diesen_begã¼nstigten: string (nullable = true)
 |-- betrag_je_vorhaben_im_rahmen_des_eler_(eu_mittel): string (nullable = true)
 |-- eler_gesamtbetrag_fã¼r_diesen_begã¼nstigten_(eu_mittel): string (nullable = true)
 |-- betrag_je_vorhab

In [5]:
# let's know the number of non null or nan values present in each column

# 1. Build expressions dynamically based on each column's specific data type
count_expressions = []


for col_name, col_type in Germany_23.dtypes:
    # Base condition: works for ALL data types (Timestamps, Strings, Ints, etc.)
    condition = F.col(col_name).isNotNull()

    # Only append the NaN check if the column is a floating-point numeric type
    if col_type in ("double", "float"):
        condition = condition & ~F.isnan(F.col(col_name))
    
    # Rule B: Catch empty text cells in string columns
    elif col_type == "string":
        condition = condition & ~(F.trim(F.col(col_name))).isin("", "N/A", "n/a", "NA", "na")
    
    # Aggregate using the safe conditional block
    count_expressions.append(F.count(F.when(condition, 1)).alias(col_name))

# 2. Run the single, optimized aggregation across the cluster
counts_row = Germany_23.select(count_expressions).first()

print(counts_row)


Row(haushaltsjahr=1393345, name_des_begã¼nstigten_rechtstrã¤gers_verdands=1393345, wenn_teil_einer_gruppe:_name_des_mutterunternehmens=1316, wenn_teil_einer_gruppe:_steuerliches_identifikationsmerkmal=1285, plz=1294986, gemeinde=1393345, betroffener_staat=1393345, code_der_maãnahme_der_interventionskategorie_des_sektors_gemã¤ã_anhang_ix=1393345, spezifisches_ziel=1865, anfangsdatum=2339, enddatum=9233, betrag_je_vorhaben_im_rahmen_des_egfl=1108762, egfl__gesamtbetrag_fã¼r_diesen_begã¼nstigten=1393345, betrag_je_vorhaben_im_rahmen_des_eler_(eu_mittel)=284560, eler_gesamtbetrag_fã¼r_diesen_begã¼nstigten_(eu_mittel)=284583, betrag_je_vorhaben_im_rahmen_der_nationalen_kofinanzierung=18, national_kofinanzierter_gesamtbetrag_fã¼r_diesen_begã¼nstigten=284583, summe_des_eler_betrags_(eu_mittel)_und_des_kofinanzierten_betrags=284583, eu_betrag_(egfl_und_eler)_und_kofinanzierter_betrag_insgesamt_fã¼r_diesen_begã¼nstigten*=1393345, source_country=1393345, source_year=1393345, ingested_at=139334

In [6]:
# Count total rows in the DataFrame
total_rows = Germany_23.count()

# Print header
print(f'{"Column Name": <65} | {"Missing Percentage"}')
print("-" * 85)

# Calculate and print missing percentage for each column
for column, valid_count in counts_row.asDict().items():
    missing_percentage = ((total_rows - valid_count) / total_rows) * 100
    print(f"{column: <65} | {missing_percentage: >10.3f}%")


Column Name                                                       | Missing Percentage
-------------------------------------------------------------------------------------
haushaltsjahr                                                     |      0.000%
name_des_begã¼nstigten_rechtstrã¤gers_verdands                    |      0.000%
wenn_teil_einer_gruppe:_name_des_mutterunternehmens               |     99.906%
wenn_teil_einer_gruppe:_steuerliches_identifikationsmerkmal       |     99.908%
plz                                                               |      7.059%
gemeinde                                                          |      0.000%
betroffener_staat                                                 |      0.000%
code_der_maãnahme_der_interventionskategorie_des_sektors_gemã¤ã_anhang_ix |      0.000%
spezifisches_ziel                                                 |     99.866%
anfangsdatum                                                      |     99.832%
enddatum         

In [2]:
Germany_23.select(
   
    "name_des_begã¼nstigten_rechtstrã¤gers_verdands", 
    "code_der_maãnahme_der_interventionskategorie_des_sektors_gemã¤ã_anhang_ix",
    "betrag_je_vorhaben_im_rahmen_des_egfl",
    "egfl__gesamtbetrag_fã¼r_diesen_begã¼nstigten",
    "eler_gesamtbetrag_fã¼r_diesen_begã¼nstigten_(eu_mittel)",
    "eu_betrag_(egfl_und_eler)_und_kofinanzierter_betrag_insgesamt_fã¼r_diesen_begã¼nstigten*"
).show(12, truncate= False)

NameError: name 'Germany_24' is not defined

In [9]:
Germany_23.select(
   
    "name_des_begã¼nstigten_rechtstrã¤gers_verdands", 
    "betrag_je_vorhaben_im_rahmen_des_egfl",
    "egfl__gesamtbetrag_fã¼r_diesen_begã¼nstigten",
    "eler_gesamtbetrag_fã¼r_diesen_begã¼nstigten_(eu_mittel)",
    "eu_betrag_(egfl_und_eler)_und_kofinanzierter_betrag_insgesamt_fã¼r_diesen_begã¼nstigten*"
).distinct().show(12, truncate= False)

+-----------------------------------------------------------+-------------------------------------+--------------------------------------------+-------------------------------------------------------+----------------------------------------------------------------------------------------+
|name_des_begã¼nstigten_rechtstrã¤gers_verdands             |betrag_je_vorhaben_im_rahmen_des_egfl|egfl__gesamtbetrag_fã¼r_diesen_begã¼nstigten|eler_gesamtbetrag_fã¼r_diesen_begã¼nstigten_(eu_mittel)|eu_betrag_(egfl_und_eler)_und_kofinanzierter_betrag_insgesamt_fã¼r_diesen_begã¼nstigten*|
+-----------------------------------------------------------+-------------------------------------+--------------------------------------------+-------------------------------------------------------+----------------------------------------------------------------------------------------+
|A. & G. BÃ¼ning GbR                                        |3739.68                              |13548.35                       

In [6]:
Germany_23_cleaned= Germany_23.select(
    F.col("name_des_begã¼nstigten_rechtstrã¤gers_verdands").alias("beneficiary"),
    F.col("gemeinde").alias("municipality"),
    F.col("source_country").alias("country"),
    F.col("source_year").alias("year"),
    F.col("code_der_maãnahme_der_interventionskategorie_des_sektors_gemã¤ã_anhang_ix").alias("intervention_code"),
    F.col("betrag_je_vorhaben_im_rahmen_des_egfl").cast(DoubleType()).alias("total_eagf_income_support"),
    F.col("betrag_je_vorhaben_im_rahmen_des_eler_(eu_mittel)").cast(DoubleType()).alias("total_eafrd_income_support"),
    F.col("betrag_je_vorhaben_im_rahmen_der_nationalen_kofinanzierung").cast(DoubleType()).alias("national_cofunding_amount")
    
).fillna(0.0, subset=["total_eagf_income_support", "total_eafrd_income_support", "national_cofunding_amount"])

# fill nulls in text fields
Germany23_cleaned= Germany_23_cleaned.fillna("UNKNOWN", subset= ["beneficiary", "municipality", "intervention_code"])

In [7]:
Germany23_cleaned.show(5, truncate=False)

+---------------------+------------+-------+----+-----------------+-------------------------+--------------------------+-------------------------+
|beneficiary          |municipality|country|year|intervention_code|total_eagf_income_support|total_eafrd_income_support|national_cofunding_amount|
+---------------------+------------+-------+----+-----------------+-------------------------+--------------------------+-------------------------+
|A & A Grebenstein GbR|Seevetal    |GERMANY|2023|II.6             |1759.17                  |0.0                       |0.0                      |
|A & A Grebenstein GbR|Seevetal    |GERMANY|2023|II.4             |3249.72                  |0.0                       |0.0                      |
|A & A Grebenstein GbR|Seevetal    |GERMANY|2023|II.3             |1779.88                  |0.0                       |0.0                      |
|A & A Grebenstein GbR|Seevetal    |GERMANY|2023|II.1             |6658.39                  |0.0                      

In [8]:
spark.stop()